# Diagnose LLM Feature Coverage + Validity

Before widening LLM scoring, understand two things from the actual data:

**(A) Why is coverage only 5.9%** (expected ~10% = top-200 of ~1920-doc pools)? The LLM scores were
computed on `rerank_llm_feature.ipynb`'s pool; the ensemble rebuilds the pool independently. If the
two pools drift, some LLM-scored docs fall outside the ensemble's pool and are wasted. This measures
the drift and localizes the cause (does the pool construction actually reproduce?).

**(B) Is the LLM feature meaningful?** Do judged-relevant trials get higher LLM yes/no scores than
non-relevant ones? If not, its #2 feature-importance is the LTR latching onto noise, and widening it
would be chasing a number. We check score separation by relevance label and a rank-AUC.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q rank-bm25 sentence-transformers datasets numpy tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, pickle, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from ctmatch.evaluation.eval_utils import load_eval_datasets

DATA_ROOT = '/content/drive/MyDrive/ct_data23'
os.environ['CTMATCH_DATA_ROOT'] = DATA_ROOT
FULLTEXT_CORPUS = f'{DATA_ROOT}/doc_texts_fulltext.txt'
DENSE_EMB_FILE  = f'{DATA_ROOT}/doc_embeddings_retriever-v2_fulltext.npy'
BM25_CACHE      = f'{DATA_ROOT}/bm25_fulltext.pkl'
LLM_SCORES      = f'{DATA_ROOT}/llm_reranker_scores.jsonl'
CAND_K, LLM_TOP_K = 1000, 200

_idx = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
corpus_ids = [r['text'].strip() for r in _idx]
id2corpus_idx = {d: i for i, d in enumerate(corpus_ids)}
with open(BM25_CACHE, 'rb') as f:
    bm25 = pickle.load(f)
doc_emb = np.load(DENSE_EMB_FILE).astype(np.float32)
doc_emb /= (np.linalg.norm(doc_emb, axis=1, keepdims=True) + 1e-9)
q_encoder = SentenceTransformer('semaj83/ctmatch-retriever-v2')

all_sets = load_eval_datasets(f'{DATA_ROOT}/evaluation/trec_data', f'{DATA_ROOT}/evaluation/kz_data')

# precomputed LLM scores, grouped by (source, topic)
llm_by_topic = {}
with open(LLM_SCORES) as f:
    for line in f:
        r = json.loads(line)
        llm_by_topic.setdefault((r['source'], r['topic_id']), {})[r['doc_id']] = r['llm_score']
print('topics with LLM scores:', len(llm_by_topic),
      '| avg docs/topic:', round(np.mean([len(v) for v in llm_by_topic.values()]), 1))

## (A) Pool-drift diagnosis

Rebuild the candidate pool exactly as the ensemble does, and for each topic ask: of the 200 docs the
LLM scored, how many are (1) in the pool at all, and (2) still in the RRF top-200? If (1) is ~200 the
pool reproduces and the low coverage is just pool size; if (1) ≪ 200 the pool genuinely drifted.

In [ ]:
from tqdm.auto import tqdm

def pool_and_rrf(text):
    qv = q_encoder.encode(text, normalize_embeddings=True).astype(np.float32)
    bm = np.array(bm25.index.get_scores(bm25._tokenize(text)))
    dn = doc_emb @ qv
    bm_top = np.argpartition(-bm, CAND_K)[:CAND_K]
    dn_top = np.argpartition(-dn, CAND_K)[:CAND_K]
    cand = list(set(bm_top.tolist()) | set(dn_top.tolist()))
    bm_rank = {j: r for r, j in enumerate(sorted(cand, key=lambda j: -bm[j]))}
    dn_rank = {j: r for r, j in enumerate(sorted(cand, key=lambda j: -dn[j]))}
    rrf = {j: 1.0/(60+bm_rank[j]+1) + 1.0/(60+dn_rank[j]+1) for j in cand}
    top200 = [j for j, _ in sorted(rrf.items(), key=lambda x: -x[1])[:LLM_TOP_K]]
    return set(cand), set(top200), bm, dn

rows = []
sample = list(llm_by_topic.items())
for (src, tid), scored in tqdm(sample, desc='pool drift'):
    text = all_sets[src]['topic2text'].get(tid)
    if text is None:
        continue
    cand, top200, bm, dn = pool_and_rrf(text)
    scored_idx = [id2corpus_idx[d] for d in scored if d in id2corpus_idx]
    in_pool = sum(j in cand for j in scored_idx)
    in_top  = sum(j in top200 for j in scored_idx)
    rows.append((src, tid, len(scored_idx), in_pool, in_top))

arr = np.array([[r[2], r[3], r[4]] for r in rows], dtype=float)
print(f'\nPer topic (mean over {len(rows)} topics):')
print(f'  LLM-scored docs         : {arr[:,0].mean():.0f}')
print(f'  ...still IN the pool     : {arr[:,1].mean():.0f}  ({100*arr[:,1].sum()/arr[:,0].sum():.1f}%)')
print(f'  ...still in RRF top-200  : {arr[:,2].mean():.0f}  ({100*arr[:,2].sum()/arr[:,0].sum():.1f}%)')
print('\nInterpretation:')
print('  in-pool ~200  -> pool reproduces; low coverage is just pool size (widen top-K helps cleanly)')
print('  in-pool <<200 -> pool DRIFTED between runs; must find why before trusting the feature')

In [ ]:
# If docs drifted out of the pool: where do they rank NOW? A doc that was RRF-top-200 but is now
# outside the 1000-deep pool means its bm25/dense score changed a lot -> retriever inputs differ.
src, tid = rows[0][0], rows[0][1]
text = all_sets[src]['topic2text'][tid]
cand, top200, bm, dn = pool_and_rrf(text)
scored = llm_by_topic[(src, tid)]
missing = [d for d in scored if id2corpus_idx.get(d) not in cand]
print(f'Topic {src}/{tid}: {len(scored)} LLM-scored, {len(missing)} now outside the pool')
if missing:
    bm_order = np.argsort(-bm); dn_order = np.argsort(-dn)
    bm_pos = {int(j): r for r, j in enumerate(bm_order[:5000])}
    dn_pos = {int(j): r for r, j in enumerate(dn_order[:5000])}
    print('\nsample drifted docs — current full-corpus rank by each retriever:')
    for d in missing[:8]:
        j = id2corpus_idx[d]
        print(f'  {d}: bm25 rank {bm_pos.get(j, ">5000")}, dense rank {dn_pos.get(j, ">5000")}')
    print('\nIf these rank e.g. 200-1000 now (just outside top-K union) -> minor ranking jitter.')
    print('If they rank in the thousands -> the embeddings/index differ from the LLM run.')

## (B) Is the LLM feature meaningful?

Join LLM scores with qrel labels (judged docs only). If the feature is real, rel=2 > rel=1 > rel=0
in mean score, and a rel≥1 doc outscores a rel=0 doc well above chance (rank-AUC ≫ 0.5).

In [ ]:
by_label = {0: [], 1: [], 2: []}
pos, neg = [], []   # for rank-AUC within topic
auc_terms = []
for (src, tid), scored in llm_by_topic.items():
    rel = all_sets[src]['rel_dict'].get(tid, {})
    tp, tn = [], []
    for d, s in scored.items():
        if d in rel:
            by_label[rel[d]].append(s)
            (tp if rel[d] >= 1 else tn).append(s)
    # per-topic AUC: P(pos score > neg score)
    if tp and tn:
        wins = sum(p > n for p in tp for n in tn)
        auc_terms.append(wins / (len(tp) * len(tn)))

print('Mean LLM yes/no score by relevance label (judged docs in the LLM top-200):')
for l in [2, 1, 0]:
    v = by_label[l]
    print(f'  rel={l}: n={len(v):5d}  mean={np.mean(v):+.3f}  median={np.median(v):+.3f}')
print(f'\nPer-topic rank-AUC (rel>=1 vs rel=0), mean over {len(auc_terms)} topics: {np.mean(auc_terms):.3f}')
print('  0.5 = no signal (noise) | >0.65 = the LLM feature genuinely discriminates')
print('\nThis is the load-bearing check: if AUC ~0.5, the LLM feature is noise and its #2 importance')
print('is the LTR overfitting — do NOT widen it. If AUC is high, widening is well-justified.')